# Create regression input dataset for payroll model

In [2]:
import xarray as xr
import xesmf as xe
import xagg as xa 
import numpy as np
import pandas as pd
import geopandas as gpd
import xhistogram.xarray as xhist
import os
import re
import glob
import fsspec
from matplotlib import pyplot as plt
from cartopy import crs as ccrs

from numba import jit as njit

from funcs_support import utility_save,get_params,get_filepaths
from funcs_aux import _restore_doys
dir_list = get_params()
# Get filesystem
fs = fsspec.implementations.local.LocalFileSystem()

In [2]:
from distributed import Client
client = Client()
client

/glade/u/home/schwarzwald/.conda/envs/impact_uncert/lib/python3.14/site-packages/distributed/node.py:188: UserWarning: Port 8787 is already in use.
Perhaps you already have a cluster running?
Hosting the HTTP server on port 44849 instead
  warnings.warn(


Connection method: Cluster object,Cluster type: distributed.LocalCluster
Dashboard: https://jupyterhub.hpc.ucar.edu/stable/user/schwarzwald/bcd_me_proc3/proxy/44849/status,
Dashboard: https://jupyterhub.hpc.ucar.edu/stable/user/schwarzwald/bcd_me_proc3/proxy/44849/status,Workers: 6
Total threads: 18,Total memory: 72.00 GiB
Status: running,Using processes: True
Comm: tcp://127.0.0.1:42671,Workers: 0
Dashboard: https://jupyterhub.hpc.ucar.edu/stable/user/schwarzwald/bcd_me_proc3/proxy/44849/status,Total threads: 0
Started: Just now,Total memory: 0 B
Comm: tcp://127.0.0.1:44573,Total threads: 3
Dashboard: https://jupyterhub.hpc.ucar.edu/stable/user/schwarzwald/bcd_me_proc3/proxy/41003/status,Memory: 12.00 GiB
Nanny: tcp://127.0.0.1:39553,


2026-03-09 11:57:59,077 - distributed.shuffle._scheduler_plugin - WARNING - Shuffle 95c828cabb3de49d4058973178d9df21 initialized by task ('rechunk-merge-rechunk-transfer-740d09739d3334b73c5cfbd8eba2275c', 0, 0, 0, 0, 5, 0, 0, 0) executed on worker tcp://127.0.0.1:44573
2026-03-09 11:58:03,483 - distributed.shuffle._scheduler_plugin - WARNING - Shuffle 95c828cabb3de49d4058973178d9df21 deactivated due to stimulus 'task-finished-1773079083.4797156'
2026-03-09 11:58:19,779 - distributed.shuffle._scheduler_plugin - WARNING - Shuffle 95c828cabb3de49d4058973178d9df21 initialized by task ('rechunk-merge-rechunk-transfer-740d09739d3334b73c5cfbd8eba2275c', 0, 0, 0, 0, 3, 0, 0, 0) executed on worker tcp://127.0.0.1:46809
2026-03-09 11:58:22,887 - distributed.shuffle._scheduler_plugin - WARNING - Shuffle 95c828cabb3de49d4058973178d9df21 deactivated due to stimulus 'task-finished-1773079102.8842952'
2026-03-09 11:59:10,273 - distributed.shuffle._scheduler_plugin - WARNING - Shuffle 95c828cabb3de49d

In [3]:
# US Counties
gdf = gpd.read_file(dir_list['aux']+'geo_data/UScounties.shp')

# Histogram parameters
# In F
bin_width = 5
bins = np.arange(-50,131,bin_width)

params_subset = {'lat':slice(23,53),'lon':slice(-127,-60)}
# Get reference 0.25x0.25 grid that slots into the 
# ref_grid 1x1 grid perfectly 
coarse_grid = xr.Dataset(coords = {'lat':(['lat'],np.arange(params_subset['lat'].start+0.5,
    														 params_subset['lat'].stop-0.4)),
     								'lon':(['lon'],np.arange(params_subset['lon'].start+0.5,
    														 params_subset['lon'].stop-0.4))})

fine_grid = xr.Dataset(coords = {'lat':(['lat'],np.arange(params_subset['lat'].start+0.125,
														 params_subset['lat'].stop-0.124,0.25)),
								'lon':(['lon'],np.arange(params_subset['lon'].start+0.125,
														 params_subset['lon'].stop-0.124,0.25))})

var = 'tasmax'


# Load population by grid cell
pop_fn = dir_list['aux']+'pop_fx_LANDSCAN_conus_day_025deg_2016.nc'
if not fs.exists(pop_fn):
    pop = xr.open_dataset(dir_list['aux']+'pop_data/landscan-usa-2016-conus-day.tif')
    pop = xa.fix_ds(pop.isel(band=0).rename({'band_data':'pop'})).coarsen({'lat':300,'lon':300}).sum()
    pop['lat'] = np.round(pop['lat'],3) # floating point errors otehrwise
    pop['lon'] = np.round(pop['lon'],3)
    pop.attrs['SOURCE'] = '1c_create_reginput_payroll.ipynb'
    pop.attrs['DESCRIPTION'] = 'landscan USA 2016 daytime CONUS data, coarsened (by summing) 300x to a 0.25-deg grid'
    utility_save(pop,pop_fn)
else:
    pop = xr.open_dataset(pop_fn)


In [11]:
# Get weightmap between reference grid and counties
wm = {'fine':{'raw':xa.pixel_overlaps(fine_grid,gdf),
      'pop':xa.pixel_overlaps(fine_grid,gdf,weights=pop.pop)},
      'coarse':{'raw':xa.pixel_overlaps(coarse_grid,gdf),
      'pop':xa.pixel_overlaps(coarse_grid,gdf,weights=pop.pop)}}

creating polygons for each pixel...
calculating overlaps between pixels and output polygons...
success!
creating polygons for each pixel...
regridding weights to data grid...


/glade/u/home/schwarzwald/.conda/envs/impact_uncert/lib/python3.14/site-packages/xagg/core.py:192: UserWarning: The `weights` input (bbox latmin, latmax, lonmin, lonmax: 24.375, 49.375, -124.875, -66.875 spans less area than the `ds` input (bbox: 23.125, 52.875, -126.875, -60.125). Weights beyond  this bounding box may be set to 0 or incorrectly interpolated.
  warnings.warn('The `weights` input (bbox latmin, latmax, lonmin, lonmax: '+


calculating overlaps between pixels and output polygons...
success!
creating polygons for each pixel...
lat/lon bounds not found in dataset; they will be created.
calculating overlaps between pixels and output polygons...
success!
creating polygons for each pixel...
regridding weights to data grid...


/glade/u/home/schwarzwald/.conda/envs/impact_uncert/lib/python3.14/site-packages/xagg/core.py:192: UserWarning: The `weights` input (bbox latmin, latmax, lonmin, lonmax: 24.375, 49.375, -124.875, -66.875 spans less area than the `ds` input (bbox: 23.5, 52.5, -126.5, -60.5). Weights beyond  this bounding box may be set to 0 or incorrectly interpolated.
  warnings.warn('The `weights` input (bbox latmin, latmax, lonmin, lonmax: '+


calculating overlaps between pixels and output polygons...
success!


In [5]:
@njit
def numba_histogram(data, bins):
    hist = np.zeros(len(bins) - 1, dtype=np.int64)
    for value in data:
        for i in range(len(bins) - 1):
            if bins[i] <= value < bins[i + 1]:
                hist[i] += 1
                break
    return hist

def fast_histogram(da, bins, dim):
    return xr.apply_ufunc(
        numba_histogram,
        da,
        input_core_dims=[dim],
        output_core_dims=[['bin']],
        kwargs={'bins': bins},
        vectorize=True,
        dask="parallelized",
        output_dtypes=[int],
        dask_gufunc_kwargs = {'output_sizes':{'bin':len(bins)-1}}
    )

In [17]:
df = get_filepaths()

dss_out_all = []

for obs_mod in ['ERA5-025','MERRA2','JRA-3Q','GMFD','PRISM','ERA5-025-1deg','MERRA2-1deg','JRA-3Q-1deg','GMFD-1deg','PRISM-1deg']:
    print(f'\nProcessing {obs_mod}')

    df_tmp = df.query(f'model == "{obs_mod}" and (varname == "tasmax") and freq == "day"')
    
    # Load
    print(f'Using file {df_tmp.iloc[0]['path']}...')
    ds = xr.open_dataset(df_tmp.iloc[0]['path'])

    if obs_mod == 'PRISM':
        # So high resolution, need to make sure this is a dask array 
        # with small chunks for regridding
        ds = ds.chunk({'lat':-1,'lon':-1,'time':1})
    
    # Subset to CONUS
    ds = ds.sel(**params_subset)
    
    # Regrid to 0.25 deg grid
    if '1deg' not in obs_mod:
        rgrd = xe.Regridder(ds,fine_grid,'bilinear')
        ds = rgrd(ds)
        # Make sure areas that don't overlap with destination grid
        # are nan, not 0 (T in K can't be 0)
        ds[var] = ds[var].where(ds[var]!=0)

    if obs_mod == 'PRISM':
        # Now need to rechunk such that time is one chunk
        ds = ds.chunk({'lat':20,'lon':20,'time':-1})
    
    # Reshape to dayofyear x year 
    ds = ds.assign_coords({'year':ds.time.dt.year,
                'dayofyear':ds.time.dt.dayofyear})
    
    ds = ds.set_index(time=['dayofyear','year']).unstack()

    # For some reason, this did not make dayofyear one chunk above 
    ds = ds.chunk({'lat':20,'lon':20,'dayofyear':-1,'year':-1})
    
    # Calculate histogram
    tbins = fast_histogram(ds[var],bins=(bins-32)*5/9+273.15,dim=['dayofyear'])
    tbins['bin'] = bins[0:-1]+bin_width/2
    
    # Aggregate
    with xa.set_options(impl='numba'):
        if '1deg' in obs_mod:
            agg = {typ:xa.aggregate(tbins.chunk({'lat':-1,'lon':-1,'year':1}),wm['coarse'][typ]) for typ in wm['coarse']}
        else:
            agg = {typ:xa.aggregate(tbins.chunk({'lat':-1,'lon':-1,'year':1}),wm['fine'][typ]) for typ in wm['fine']}

    # Get out in xarray dataset format
    dss_out = {typ:ag.to_dataset() for typ,ag in agg.items()}
    
    dss_out = xr.concat([dss_out[typ][var].expand_dims({'weighting':[typ],'model':[obs_mod]})
                           for typ in dss_out],
                          dim='weighting')

    dss_out_all.append(dss_out)


Processing ERA5-025
Using file /glade/derecho/scratch/schwarzwald/climate_raw_derecho/ERA5-025/tasmax_day_ERA5-025_historical_reanalysis_19810101-20241231.zarr...
adjusting grid... (this may happen because only a subset of pixels were used for aggregation for efficiency - i.e. [subset_bbox=True] in xa.pixel_overlaps())
grid adjustment successful
aggregating tasmax...


/glade/u/home/schwarzwald/.conda/envs/impact_uncert/lib/python3.14/site-packages/distributed/client.py:3375: UserWarning: Sending large graph of size 1.93 GiB.
This may cause some slowdown.
Consider loading the data with Dask directly
 or using futures or delayed objects to embed the data into the graph without repetition.
See also https://docs.dask.org/en/stable/best-practices.html#load-data-with-dask for more information.
  warnings.warn(


all variables aggregated to polygons!
adjusting grid... (this may happen because only a subset of pixels were used for aggregation for efficiency - i.e. [subset_bbox=True] in xa.pixel_overlaps())
grid adjustment successful
aggregating tasmax...


/glade/u/home/schwarzwald/.conda/envs/impact_uncert/lib/python3.14/site-packages/distributed/client.py:3375: UserWarning: Sending large graph of size 1.93 GiB.
This may cause some slowdown.
Consider loading the data with Dask directly
 or using futures or delayed objects to embed the data into the graph without repetition.
See also https://docs.dask.org/en/stable/best-practices.html#load-data-with-dask for more information.
  warnings.warn(


all variables aggregated to polygons!

Processing MERRA2
Using file /glade/derecho/scratch/schwarzwald/climate_raw_derecho/MERRA2/tasmax_day_MERRA2_historical_reanalysis_19800101-20241231.nc...
adjusting grid... (this may happen because only a subset of pixels were used for aggregation for efficiency - i.e. [subset_bbox=True] in xa.pixel_overlaps())
grid adjustment successful
aggregating tasmax...


...

all variables aggregated to polygons!

Processing PRISM-1deg
Using file /glade/derecho/scratch/schwarzwald/climate_raw_derecho/PRISM-1deg/tasmax_day_PRISM-1deg_historical_obs_19810101-20111231.zarr...
adjusting grid... (this may happen because only a subset of pixels were used for aggregation for efficiency - i.e. [subset_bbox=True] in xa.pixel_overlaps())
grid adjustment successful
aggregating tasmax...


/glade/u/home/schwarzwald/.conda/envs/impact_uncert/lib/python3.14/site-packages/distributed/client.py:3375: UserWarning: Sending large graph of size 174.75 MiB.
This may cause some slowdown.
Consider loading the data with Dask directly
 or using futures or delayed objects to embed the data into the graph without repetition.
See also https://docs.dask.org/en/stable/best-practices.html#load-data-with-dask for more information.
  warnings.warn(


all variables aggregated to polygons!
adjusting grid... (this may happen because only a subset of pixels were used for aggregation for efficiency - i.e. [subset_bbox=True] in xa.pixel_overlaps())
grid adjustment successful
aggregating tasmax...


/glade/u/home/schwarzwald/.conda/envs/impact_uncert/lib/python3.14/site-packages/distributed/client.py:3375: UserWarning: Sending large graph of size 174.75 MiB.
This may cause some slowdown.
Consider loading the data with Dask directly
 or using futures or delayed objects to embed the data into the graph without repetition.
See also https://docs.dask.org/en/stable/best-practices.html#load-data-with-dask for more information.
  warnings.warn(


all variables aggregated to polygons!


In [18]:
dss_out_all = xr.concat(dss_out_all,dim='model')

# Add back in other metadata 
other_vars = agg['raw'].to_dataset()
other_vars = other_vars[[v for v in other_vars if v != var]]
dss_out_all = xr.merge([dss_out_all.to_dataset(name=var+'bins'),
                        other_vars])

/glade/derecho/scratch/schwarzwald/tmp/ipykernel_66106/306083719.py:1: FutureWarning: In a future version of xarray the default value for join will change from join='outer' to join='exact'. This change will result in the following ValueError: cannot be aligned with join='exact' because index/labels/sizes are not equal along these coordinates (dimensions): 'year' ('year',) The recommendation is to set join explicitly for this case.
  dss_out_all = xr.concat(dss_out_all,dim='model')


In [37]:
# Load sample dataset
# (originally "Replication_data.csv" from hle_repl_ab/Data, uploaded and renamed)
df_sample = pd.read_csv(dir_list['aux']+'hle_replication_data.csv')

In [39]:
# Subset to just the relevant bins + index variables, renamed as in sample df
dss_out_all = dss_out_all.rename({'FIPS':'fips','STATE_FIPS':'state'})[[var+'bins','fips','state']]
# Turn into int fips to match sample df
for v in ['fips','state']:
    dss_out_all[v] = dss_out_all[v].astype(int)

In [63]:
for weighting in dss_out_all.weighting.values:
    # Extract data for model / weighting scheme
    ds_tmp = dss_out_all.sel(weighting = weighting,drop=True)
    
    # Get binwidths (for bin column naming)
    ds_tmp['bindiffs'] = ds_tmp['bin'].diff('bin')
    
    # Sum below 40 and above 90 to their own variables
    ds_tmp['days90plus_dry'] = ds_tmp[var+'bins'].sel(bin = slice(90,None)).sum('bin')
    ds_tmp['days95plus_dry'] = ds_tmp[var+'bins'].sel(bin = slice(95,None)).sum('bin')
    ds_tmp['days100plus_dry'] = ds_tmp[var+'bins'].sel(bin = slice(100,None)).sum('bin')
    ds_tmp['below40_dry'] = ds_tmp[var+'bins'].sel(bin = slice(None,40)).sum('bin')
    # Subset to remaining bins
    ds_tmp = ds_tmp.sel(bin=slice(40,100))
    
    # Subset to just needed years
    ds_tmp = ds_tmp.sel(year = np.unique(df_sample.year))
    
    # Change bin dimension to match sample df column names
    ds_tmp['bin'] = [str(int(b-diff/2))+'to'+str(int(b+diff/2)) for b,diff in zip(ds_tmp['bin'],ds_tmp['bindiffs'])]
    
    # Put into dataframe format
    df_bins = (ds_tmp[['fips',var+'bins']].to_dataframe().reset_index().
                 drop('poly_idx',axis=1).
                 set_index(['fips','year','model']))
    df_bins = df_bins.pivot(columns = 'bin',values=var+'bins')
    df_bins.columns = ['FND_'+col+'_dry' for col in df_bins.columns]
    
    # Add 90+, 40- in
    df_bins = df_bins.join((ds_tmp[['fips','days90plus_dry','days95plus_dry','days100plus_dry','below40_dry']].to_dataframe().reset_index().
     drop('poly_idx',axis=1).
     set_index(['fips','year','model'])))
    
    # Add other variables, metadata
    df_out = (df_sample[['fips','year','prec_dry','state','lny_nonag',
                        'Dividends','pc_Div','mean_Div4','mean_Div2','pc_DivL1']].
              set_index(['fips','year']).join(df_bins))
    
    # Rename to match standards
    df_out = df_out.rename_axis(index={'model': 'obs_mod'})

    # Remove rows without data for an obs product
    df_out = df_out.iloc[~df.FND_95to100_dry.isnull()]
    
    # Split out by 0.25 deg and 1 deg inputs 
    df_out_025deg = df_out.iloc[['1deg' not in m for m in df_out.index.get_level_values('obs_mod')],:]
    df_out_1deg = df_out.iloc[['1deg' in m for m in df_out.index.get_level_values('obs_mod')],:]
    
    # Save
    df_out_025deg.to_csv(f'{dir_list['proc']}reg_inputs/payroll/reginputs_payroll_025deg_{weighting}wght.csv')
    df_out_1deg.to_csv(f'{dir_list['proc']}reg_inputs/payroll/reginputs_payroll_1deg_{weighting}wght.csv')

In [16]:
for weighting in ['raw', 'pop']:
    for res in ['025deg','1deg']:
        fn = f'{dir_list['proc']}reg_inputs/payroll/reginputs_payroll_{res}_{weighting}wght.csv'
        df = pd.read_csv(fn)
        df = df.iloc[~df.FND_95to100_dry.isnull()]
        df.to_csv(fn)

In [15]:

df_out_025deg = df_out_025deg.iloc[~df.FND_95to100_dry.isnull()]

,fips,year,obs_mod,prec_dry,state,lny_nonag,Dividends,pc_Div,mean_Div4,mean_Div2,...,FND_70to75_dry,FND_75to80_dry,FND_80to85_dry,FND_85to90_dry,FND_90to95_dry,FND_95to100_dry,days90plus_dry,days95plus_dry,days100plus_dry,below40_dry
0,1001,1986,ERA5-025,3.398327,1,0.878577,NaN,NaN,NaN,NaN,...,48.754799,36.325911,39.050323,46.831639,54.951190,26.272919,82.792907,27.841718,1.568798,0.088655
1,1001,1986,GMFD,3.398327,1,0.878577,NaN,NaN,NaN,NaN,...,42.374939,36.036179,36.714834,65.898624,51.873101,16.109155,72.789923,20.916821,4.807666,0.896554
2,1001,1986,JRA-3Q,3.398327,1,0.878577,NaN,NaN,NaN,NaN,...,51.614187,28.829029,62.466500,78.026916,24.342672,3.081590,27.424263,3.081590,0.000000,0.915525
3,1001,1986,MERRA2,3.398327,1,0.878577,NaN,NaN,NaN,NaN,...,39.935376,35.281380,20.531116,49.018716,47.954836,42.141528,101.580063,53.625228,11.483700,1.000000
4,1001,1986,PRISM,3.398327,1,0.878577,NaN,NaN,NaN,NaN,...,41.097896,43.336057,29.532261,56.870131,65.884049,21.110108,86.994157,21.110108,0.000000,2.000000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
399529,56045,2010,PRISM,1.173621,56,1.890793,2577.169624,0.779779,0.800208,0.818822,...,30.164313,26.854961,21.931188,25.766677,13.872684,4.682349,18.951496,5.078812,0.396463,88.619304
399530,56045,2011,ERA5-025,1.258269,56,2.020477,3517.232594,1.026330,0.800208,0.818822,...,17.931161,22.549224,31.725693,22.614081,14.563051,4.491773,19.459398,4.896347,0.404575,89.871474
399532,56045,2011,JRA-3Q,1.258269,56,2.020477,3517.232594,1.026330,0.800208,0.818822,...,27.757014,29.882362,26.371887,21.241656,8.740759,0.247625,8.988384,0.247625,0.000000,91.470716
399533,56045,2011,MERRA2,1.258269,56,2.020477,3517.232594,1.026330,0.800208,0.818822,...,14.758212,16.390648,31.036235,26.380172,21.908844,13.860252,36.198512,14.289668,0.429415,82.110219


In [14]:
df

,fips,year,obs_mod,prec_dry,state,lny_nonag,Dividends,pc_Div,mean_Div4,mean_Div2,...,FND_70to75_dry,FND_75to80_dry,FND_80to85_dry,FND_85to90_dry,FND_90to95_dry,FND_95to100_dry,days90plus_dry,days95plus_dry,days100plus_dry,below40_dry
0,1001,1986,ERA5-025,3.398327,1,0.878577,NaN,NaN,NaN,NaN,...,48.754799,36.325911,39.050323,46.831639,54.951190,26.272919,82.792907,27.841718,1.568798,0.088655
1,1001,1986,GMFD,3.398327,1,0.878577,NaN,NaN,NaN,NaN,...,42.374939,36.036179,36.714834,65.898624,51.873101,16.109155,72.789923,20.916821,4.807666,0.896554
2,1001,1986,JRA-3Q,3.398327,1,0.878577,NaN,NaN,NaN,NaN,...,51.614187,28.829029,62.466500,78.026916,24.342672,3.081590,27.424263,3.081590,0.000000,0.915525
3,1001,1986,MERRA2,3.398327,1,0.878577,NaN,NaN,NaN,NaN,...,39.935376,35.281380,20.531116,49.018716,47.954836,42.141528,101.580063,53.625228,11.483700,1.000000
4,1001,1986,PRISM,3.398327,1,0.878577,NaN,NaN,NaN,NaN,...,41.097896,43.336057,29.532261,56.870131,65.884049,21.110108,86.994157,21.110108,0.000000,2.000000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
399530,56045,2011,ERA5-025,1.258269,56,2.020477,3517.232594,1.02633,0.800208,0.818822,...,17.931161,22.549224,31.725693,22.614081,14.563051,4.491773,19.459398,4.896347,0.404575,89.871474
399531,56045,2011,GMFD,1.258269,56,2.020477,3517.232594,1.02633,0.800208,0.818822,...,NaN,NaN,NaN,NaN,NaN,NaN,0.000000,0.000000,0.000000,0.000000
399532,56045,2011,JRA-3Q,1.258269,56,2.020477,3517.232594,1.02633,0.800208,0.818822,...,27.757014,29.882362,26.371887,21.241656,8.740759,0.247625,8.988384,0.247625,0.000000,91.470716
399533,56045,2011,MERRA2,1.258269,56,2.020477,3517.232594,1.02633,0.800208,0.818822,...,14.758212,16.390648,31.036235,26.380172,21.908844,13.860252,36.198512,14.289668,0.429415,82.110219
